# FlashSpec Colab + Google Drive 运行脚本

运行前先在 Colab 菜单选择 `Runtime -> Change runtime type -> GPU`，如果要验证 A100，硬件加速器选择 A100。

这个 notebook 会把代码仓库放在 Google Drive：`/content/drive/MyDrive/FlashSpecColab/repo`，benchmark JSON 结果保存在：`/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。这样 Colab runtime 重启后，代码和结果仍然保留在云盘里。

## 0. 挂载 Google Drive

第一次运行会弹出授权。授权后，Colab 可以读写你的 Google Drive。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FlashSpecColab")
PROJECT_DIR = DRIVE_ROOT / "repo"
RESULTS_DIR = DRIVE_ROOT / "results" / "colab_kernels"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. 检查 Colab GPU 环境

如果这里报错，说明当前 runtime 不是 GPU，需要先切换 Colab runtime。

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("当前 Colab runtime 没有 CUDA GPU，请先切换到 GPU/A100 runtime。")

## 2. 在 Google Drive 中拉取代码并安装依赖

如果云盘里已经有 `/content/drive/MyDrive/FlashSpecColab/repo/.git`，就执行 `git pull --ff-only`；否则 clone 到云盘。`.[triton]` 会安装 Triton 可选依赖，用于运行真实 Kernel 1。

In [ ]:
import os
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/FlashSpecColab")
PROJECT_DIR = DRIVE_ROOT / "repo"
RESULTS_DIR = DRIVE_ROOT / "results" / "colab_kernels"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/honey-floria/FlashSpec.git", str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(
        f"{PROJECT_DIR} 已存在但不是 Git 仓库。请删除/重命名这个目录，"
        "或者把 PROJECT_DIR 改成一个新的空目录。"
    )

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

%pip install -e ".[triton]"

## 3. 检查 FlashSpec、CUDA 和 Triton 状态

`HAS_TRITON=True` 且 `cuda available=True` 时，`triton_fused` 和 `triton_paged` benchmark 会走真实 Triton kernel。

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FlashSpecColab/repo")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from flashspec.triton_kernels import HAS_TRITON

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("HAS_TRITON:", HAS_TRITON)

## 3.5 定位 / 安装 Nsight Compute (ncu)

`ncu` 用于采集实测 DRAM 带宽、占用率、SM 利用率，回填 `measured_*` 字段（`--profile-ncu`）。Colab 的 CUDA 安装通常自带 ncu；找不到会尝试 apt 安装。即使没有 ncu，benchmark 仍会产出估算带宽，只是 `measured_*` 为空。

In [ ]:
import os
import shutil
import subprocess

# 优先用 PATH 里的 ncu；否则找 CUDA 默认安装路径；再否则尝试 apt 安装。
NCU_BIN = shutil.which("ncu")
if NCU_BIN is None:
    for cand in ["/usr/local/cuda/bin/ncu", "/opt/nvidia/nsight-compute/ncu"]:
        if os.path.exists(cand):
            NCU_BIN = cand
            break
if NCU_BIN is None:
    print("未找到 ncu，尝试 apt 安装 nsight-compute ...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "nsight-compute"], check=False)
    NCU_BIN = shutil.which("ncu") or "/usr/local/cuda/bin/ncu"

# 把 ncu 所在目录加进 PATH，方便后续 shell cell 直接调用。
if os.path.exists(NCU_BIN):
    os.environ["PATH"] = os.path.dirname(NCU_BIN) + os.pathsep + os.environ.get("PATH", "")
    ver = subprocess.run([NCU_BIN, "--version"], capture_output=True, text=True)
    print("NCU_BIN:", NCU_BIN)
    print(ver.stdout.splitlines()[0] if ver.stdout else ver.stderr[:200])
    HAS_NCU = True
else:
    print("仍未找到 ncu；--profile-ncu 会跳过实测字段（只保留估算带宽）。")
    HAS_NCU = False

# 供后续 cell 通过 {NCU_BIN} 插值使用。
os.environ["NCU_BIN"] = NCU_BIN if os.path.exists(NCU_BIN) else "ncu"

## 4. 运行正确性测试

这里会包含 CUDA + Triton 可用时的 Kernel 1 / Kernel 2 correctness 测试：输出对齐 PyTorch reference，并检查 `materializes_dense_kv == 0.0`。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo
!python -m unittest discover -s tests

## 5. Kernel microbenchmark 对比

- `dense`: dense KV reference attention。
- `fused`: portable PyTorch INT8 KV 路径，会先 materialize dense KV。
- `triton_fused`: 真实 Kernel 1，直接读取 INT8 K/V，在 Triton kernel 内完成反量化、QK、softmax 和 PV。
- `triton_paged`: 真实 Kernel 2，在 Triton kernel 内读取 `block_table` 并间接寻址 physical INT8 KV block。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo

# 小规模 smoke benchmark，先确认三条路径都能跑通。
!python benchmarks/microbench.py --backend dense --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --warmup 3 --repeats 5 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --warmup 3 --repeats 5 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend triton_fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --warmup 3 --repeats 5 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend paged --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --warmup 3 --repeats 5 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend triton_paged --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --warmup 3 --repeats 5 --device cuda --dtype float16 --json

## 6. A100 目标 shape benchmark

TODO P0.1/P0.2 要覆盖 `head_dim=64/128` 和 `seq_len=512/2048/4096`。下面跑 Kernel 1 Triton fused 和 Kernel 2 Triton paged 主路径，并把 JSON 保存到 Google Drive 的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。如果显存或 Colab 时间不够，可以减少 `--batch` 或 `--iters`。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab/repo"
OUT = "/content/drive/MyDrive/FlashSpecColab/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get("NCU_BIN", "ncu")

# 目标 shape 网格：head_dim=64/128 × seq_len=512/2048/4096，两条 Triton 主路径。
SHAPES = [(s, d) for d in (64, 128) for s in (512, 2048, 4096)]
BACKENDS = ["triton_fused", "triton_paged"]

for backend in BACKENDS:
    for seq, hd in SHAPES:
        out = f"{OUT}/{backend}_s{seq}_d{hd}.json"
        cmd = [
            "python", "benchmarks/microbench.py", "--backend", backend,
            "--batch", "16", "--heads", "32", "--seq-len", str(seq),
            "--head-dim", str(hd), "--block-size", "16",
            "--iters", "50", "--warmup", "10", "--repeats", "20",
            "--device", "cuda", "--dtype", "float16", "--json", "--include-raw",
            # --profile-ncu：用 ncu 采集实测带宽/占用率并回填 measured_*（按 kernel 名过滤）。
            "--profile-ncu", "--ncu-bin", NCU_BIN,
            "--output", out,
        ]
        print(">>", backend, f"s{seq} d{hd}")
        subprocess.run(cmd, check=True)

## 7. dense / portable fused / Triton fused 保存对比

这组命令用于同一个 shape 下对比 dense、portable quant 和 Triton quant 路径。重点看 `latency_ms`、`latency_p50_ms`、`latency_p90_ms`、`tokens_per_second`、`compression_ratio`、`materializes_dense_kv` 和 `latency_breakdown`。所有 JSON 都会保存到 Google Drive。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab/repo"
OUT = "/content/drive/MyDrive/FlashSpecColab/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get("NCU_BIN", "ncu")

# 同一 shape (s2048/d128) 下对比 5 条路径。dense / triton_* 走 --profile-ncu 拿实测；
# portable fused/paged 的瓶颈是 materialize dense KV（多 kernel），不做单 kernel profile。
JOBS = [
    ("dense", "dense_s2048_d128.json", True),
    ("fused", "portable_fused_s2048_d128.json", False),
    ("triton_fused", "triton_fused_s2048_d128_compare.json", True),
    ("paged", "portable_paged_s2048_d128.json", False),
    ("triton_paged", "triton_paged_s2048_d128_compare.json", True),
]
for backend, fname, profile in JOBS:
    cmd = [
        "python", "benchmarks/microbench.py", "--backend", backend,
        "--batch", "16", "--heads", "32", "--seq-len", "2048",
        "--head-dim", "128", "--block-size", "16",
        "--iters", "50", "--warmup", "10", "--repeats", "20",
        "--device", "cuda", "--dtype", "float16", "--json", "--include-raw",
        "--output", f"{OUT}/{fname}",
    ]
    if profile:
        cmd += ["--profile-ncu", "--ncu-bin", NCU_BIN]
    print(">>", backend, "profile=", profile)
    subprocess.run(cmd, check=True)

## 8. 生成 profiling report

`profile_report.py` 会读取 microbenchmark JSON，把 `nsight_compute_command`、`latency_breakdown` 和 `measured_*` profiler 字段整理成 Markdown。报告也会保存到 Google Drive。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo
!python scripts/profile_report.py /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s2048_d128_compare.json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s2048_d128_profile.md
!python scripts/profile_report.py /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s2048_d128_compare.json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s2048_d128_profile.md

## 9. 汇总云盘中的结果

读取 Google Drive 上的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/*.json`，快速查看每个 benchmark 的关键字段。

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

def g(d, k, nd=4):
    v = d.get(k)
    return round(float(v), nd) if isinstance(v, (int, float)) else v

for path in sorted(RESULTS_DIR.glob("*.json")):
    d = json.loads(path.read_text())
    est = g(d, "estimated_effective_quant_kv_bandwidth_gbps", 1)
    meas = g(d, "measured_achieved_bandwidth_gbps", 1)
    print(
        path.name,
        "| backend=", d.get("backend"),
        "| lat_ms=", g(d, "latency_ms"),
        "| tok/s=", g(d, "tokens_per_second", 1),
        "| est_bw=", est,
        "| meas_bw=", meas,
        "| occ%=", g(d, "measured_occupancy_pct", 1),
        "| sm%=", g(d, "measured_sm_utilization_pct", 1),
        "| kernels=", d.get("measured_ncu_kernel_names"),
    )
    if d.get("profiler_warning"):
        print("   [WARN]", d["profiler_warning"])
    if d.get("profiler_error"):
        print("   [ERR ]", d["profiler_error"])

## 10. 生成对比图（含实测带宽）

用 `analyze_results.py` 汇总云盘里的 JSON，输出后端对比柱状图、Triton scaling 图和 `summary.csv`。有 `--profile-ncu` 回填的实测带宽时，scaling 图会自动用实测值并在图例标 `[measured]`。

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display

REPO = "/content/drive/MyDrive/FlashSpecColab/repo"
RESULTS_DIR = "/content/drive/MyDrive/FlashSpecColab/results/colab_kernels"
ANALYSIS_DIR = f"{RESULTS_DIR}/analysis"
os.chdir(REPO)

!python scripts/analyze_results.py --results-dir "{RESULTS_DIR}" --output-dir "{ANALYSIS_DIR}"

for png in ["backend_comparison_s2048_d128.png", "triton_scaling.png"]:
    p = Path(ANALYSIS_DIR) / png
    if p.exists():
        display(Image(filename=str(p)))

## 11. Split-K A/B 测试（隔离 Split-K 的纯贡献）

`num_warps` 已固定为 4，这里只切换 Split-K 开/关：

- **OFF**：`FLASHSPEC_NUM_SPLITS=1` 强制走单 kernel 快路径；
- **ON**：不设该变量，走自适应 S（s2048→4，s4096→8）。

两次唯一变量就是 Split-K，延迟差即 Split-K 的纯贡献。运行下面两个 cell：第一个跑 4 次 benchmark，第二个打对比表。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab/repo"
OUT = "/content/drive/MyDrive/FlashSpecColab/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)

def run_ab(seq, num_splits_override):
    # num_splits_override=None -> ON(自适应)；="1" -> OFF(强制单 kernel)
    tag = "splitoff" if num_splits_override == "1" else "spliton"
    out = f"{OUT}/ab_s{seq}_{tag}.json"
    env = dict(os.environ)
    if num_splits_override is not None:
        env["FLASHSPEC_NUM_SPLITS"] = num_splits_override
    else:
        env.pop("FLASHSPEC_NUM_SPLITS", None)
    cmd = [
        "python", "benchmarks/microbench.py", "--backend", "triton_fused",
        "--batch", "16", "--heads", "32", "--seq-len", str(seq),
        "--head-dim", "128", "--block-size", "16",
        "--iters", "50", "--warmup", "10", "--repeats", "20",
        "--device", "cuda", "--dtype", "float16", "--json", "--output", out,
    ]
    print(f">> s{seq} {tag}")
    subprocess.run(cmd, check=True, env=env)

# s2048 和 s4096，各跑 OFF 和 ON
for seq in (2048, 4096):
    run_ab(seq, "1")    # OFF
    run_ab(seq, None)   # ON
print("done")

In [ ]:
import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

def lat(d):
    return d.get("measured_kernel_latency_ms") or d["latency_ms"]

print(f"{'seq':>6} {'OFF ms':>9} {'ON ms':>9} {'splits':>7} {'speedup':>8}  verdict")
for seq in (2048, 4096):
    off = json.loads((OUT / f"ab_s{seq}_splitoff.json").read_text())
    on = json.loads((OUT / f"ab_s{seq}_spliton.json").read_text())
    lo, ln = lat(off), lat(on)
    sp = lo / ln
    v = "Split-K 更快" if sp > 1.03 else ("持平" if sp > 0.97 else "Split-K 更慢")
    # splits 应为 4(s2048)/8(s4096)，坐实 Split-K 确实生效
    print(f"{seq:>6} {lo:>9.4f} {ln:>9.4f} {str(on.get('num_splits')):>7} {sp:>7.2f}x  {v}")